# Setup

In [22]:
%load_ext autoreload
%autoreload 2

## IMPORTS
import os

import geopandas as gpd
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from shapely.geometry import Point, box

import src.data_filepaths as data_filepaths
import src.utils.loaders as loaders
import src.utils.vis.colors_utils as colors_utils
from src.utils.df_tools import map_values_b_to_bp
from src.utils.loaders import process_loaded_building_df
from src.utils.vis.map import CategoricalColorMap, MapExplorer
from src.utils.vis.map_special import build_map_point_numeric_and_category

# configs
sns.set_theme()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
ALLOW_EXPORT = False

## Previous Steps

Pre-processing

This takes about 25s

In [23]:
b_gdf = process_loaded_building_df(gpd.read_parquet(data_filepaths.OVERTURE_BUILDING))
bp_gdf = process_loaded_building_df(
    gpd.read_parquet(data_filepaths.OVERTURE_BUILDING_PART)
)

osm_landuse_gdf = loaders.load_osm_landuse_1_geojson_to_gdf(data_filepaths.OSM_landuse)
mp_landuse_gdf = loaders.load_masterplan_landuse_geojson_to_gdf(
    data_filepaths.MASTERPLAN_LANDUSE
)

### Merge with Landuse, then perform Building Removal

(only adds removal reasons for now)

In [24]:
def merge_with_landuse(
    b_gdf: gpd.GeoDataFrame,
    # bp_gdf: gpd.GeoDataFrame,
):
    """
    Returns `buildings_osm_merge`, `buildings_mp_merge`
    """

    buildings_osm_merge = gpd.sjoin(
        b_gdf[["centroid"]].set_geometry("centroid"),
        osm_landuse_gdf,
        how="left",
        predicate="intersects",
    )
    # deduplicate buildings having more than one associated plot/area
    buildings_osm_merge = buildings_osm_merge[
        ~buildings_osm_merge.index.duplicated(keep="first")
    ]

    buildings_osm_merge = buildings_osm_merge.drop(columns="index_right")

    buildings_mp_merge = gpd.sjoin(
        b_gdf[["centroid"]].set_geometry("centroid"),
        mp_landuse_gdf,
        how="left",
        predicate="intersects",
    )
    # deduplicate buildings having more than one associated plot/area
    buildings_mp_merge = buildings_mp_merge[
        ~buildings_mp_merge.index.duplicated(keep="first")
    ]

    buildings_mp_merge["landuse"].value_counts(dropna=False)

    # fix bool columns, because pandas reasons...?
    buildings_mp_merge["is_sdp"] = buildings_mp_merge["is_sdp"].astype(bool)
    buildings_mp_merge["is_landed"] = buildings_mp_merge["is_landed"].astype(bool)

    # no assigning to b_gdf is being done
    return buildings_osm_merge, buildings_mp_merge


def preprocess_remove_buildings(
    b_gdf: gpd.GeoDataFrame,
    bp_gdf: gpd.GeoDataFrame,
    buildings_osm_merge,  # [landuse]
    buildings_mp_merge,  # [landuse, gpr]
    mp_landuse_gdf,
    building_size_thresholds={
        "OSM": 60,  # this number is quite high, but for demand I think it's fine
        "Google": 400,
        "Microsoft": 400,
    },
    mp_zone_coverage_threshold=0.06,
):
    """
    Removes buildings from `b_gdf`/`bp_gdf` (mapped from `b_gdf`) and returns them with removal reasons added,
    but no removal done. Creates new columns `removal_reason`, `is_removed` for both `b_gdf` and `bp_gdf`.

    There are 10 removal reasons:
    * `Landuse-OSM`: This removes the OSM landuses:
        * `construction` - self explanatory
        * `landuse=industrial, industrial=port` - to complement the port Masterplan zonings, to achieve a more
        complete removal of those buildings. This is specifically for areas around Keppel Harbour, that have long
        been designated for future development, but as of writing in 2026 still houses active port facilities.
    * `Buildingtype-Overture` The following Overture building subtypes and classes are removed:
        * Subtypes: `transportation` (includes remaining transportation buildings, and crucially, `parking`),
            `outbuilding`.
        * Classes: `roof`, `pavilion`, `hut`, `shed`, `carport`, `bridge_structure`.
    * `Size-OSM`, `Size-Google`, `Size-Microsoft`: Buildings under a specified footprint size threshold are removed.
        The threshold will be higher for the ML-derived datasets from Google/Microsoft.
        This is specified in the parameter `building_size_thresholds`.
    * `Landuse-MP`: The following **non-demand** Masterplan zoning types are removed:
        `HEALTH & MEDICAL CARE`, `EDUCATIONAL INSTITUTION`, `PLACE OF WORSHIP`, `CIVIC & COMMUNITY INSTITUTION`,
        `PARK`, `BEACH AREA`, `SPORTS & RECREATION`, `WATERBODY`, `ROAD`, `TRANSPORT FACILITIES`,
        `MASS RAPID TRANSIT`, `LIGHT RAPID TRANSIT`, `UTILITY`, `CEMETERY`, `PORT / AIRPORT`, `SPECIAL USE`.
        * `RESERVE SITE` and `OPEN SPACE` are not removed and processed in a later step.
        * In the future, `UTILITY` should be processed as well, but that requires more specific conditions to
        exclude small-medium sized utility areas like substations, while retaining large utility facilities that have
        worker prescences.
    * `Landuse-MP_coverage`: Removes buildings in zones with less than the `mp_zone_coverage_threshold` parameter.
        Note that this uses the original set of buildings (i.e. unremoved buildings).
        Default threshold is set to 0.06, with 0.08 being a rough maximum before false positives start
        getting removed.
        * There is an exception for buildings in OSM `landuse=residential`. As certain areas containing bungalows
            (e.g. Mt Pleasant, Alexandria Pk) have so incredibly low density, they end up being false positives. This
            exception is in place for those buildings. If an area *is* tagged residential on OSM, that is likely
            correct and desirable anyway.
    * `Bbox-removed-areas`: These are custom defined areas where buildings are manually removed.
        They are typically areas slated for (or currently ongoing) future development,
        but currently have enough credible non-demand buildings in
        the zone to not be removed in earlier steps.
    * `Non-OSM_MP-res`: Non-OSM (i.e. ML-derived buildings) in residential plots. The vast majority of the time,
        these buildings are just construction or temporary buildings that have been picked up by these image
        recognition models, and OSM data is more than good enough on its own.
    * `Building_ID`: Specific `building_id`s that were specified for removal
    """

    b_gdf = b_gdf.copy()
    bp_gdf = bp_gdf.copy()

    BUILDING_ID_TO_REMOVE = [
        # external buildings next to TJPG railway station
        "7e9bd117-6097-4b9c-b34e-7dd8e4d60c52",
        "bec4be75-2f1b-4214-9ab3-426d248d31af",
        # construction things warehouse at Punggol Barat
        "1a1535e6-03a5-485d-a5ff-b30a2eb97a03",
        # Neo Tiew Estate
        "6ff006d4-10f2-4fc1-910e-d24b43b258ec",
        "273cbe65-abb1-4a73-bbbb-d1dc31f2abe8",
        "be4da109-d47e-499d-8ac9-397e6fc41c94",

    ]

    # clean removals that have no other condition
    MP_LANDUSES_TO_REMOVE = [
        "HEALTH & MEDICAL CARE",
        "EDUCATIONAL INSTITUTION",
        "PLACE OF WORSHIP",
        "CIVIC & COMMUNITY INSTITUTION",
        "PARK",
        "BEACH AREA",
        "SPORTS & RECREATION",
        "WATERBODY",
        "ROAD",
        "TRANSPORT FACILITIES",
        "MASS RAPID TRANSIT",
        "LIGHT RAPID TRANSIT",
        "UTILITY",
        "CEMETERY",
        "PORT / AIRPORT",
        "SPECIAL USE",
        # "RESERVE SITE",
        # "OPEN SPACE",
    ]
    MP_LANDUSE_REMOVAL_DICT = {
        landuse: "Landuse-MP" for landuse in MP_LANDUSES_TO_REMOVE
    }
    OVERTURE_REMOVAL_SUBTYPES = ["transportation", "outbuilding"]
    OVERTURE_REMOVAL_CLASSES = [
        "roof",
        "pavilion",
        "hut",
        "shed",
        "carport",
        "bridge_structure",
        "parking",  # some have subtype=residential; class=parking
    ]

    b_gdf[["mp_landuse"]] = buildings_mp_merge[["landuse"]]
    b_gdf[["mp_gpr"]] = buildings_mp_merge[["gpr"]]
    b_gdf[["mp_OBJECTID"]] = buildings_mp_merge[["OBJECTID"]]

    # for removing buildings in areas with less than MP_COVERAGE_THRESHOLD occupancy
    # first calculate areas of a zone
    # this makes an assumption that building footprints are fully inside their zone
    # which is not true. but I can't be bothered
    b_mp_covered = b_gdf.groupby("mp_OBJECTID")["area"].sum()
    mp_area = mp_landuse_gdf.geometry.area

    b_mp_coverage_prop = b_gdf["mp_OBJECTID"].map(b_mp_covered / mp_area)

    # bounding box removal
    # specify in 4326, convert to 3414
    REMOVAL_BBOXES = {
        "southern_islands": gpd.GeoSeries(
            [box(103.69961, 1.24601, 103.81492, 1.18839)], crs=4326
        )
        .to_crs(b_gdf.crs)
        .iloc[0],
        "turf_city": gpd.GeoSeries(
            [box(103.794161, 1.332734, 103.797605, 1.335748)], crs=4326
        )
        .to_crs(b_gdf.crs)
        .iloc[0],
        "marina_south": gpd.GeoSeries(
            [box(103.859317, 1.267283, 103.872846, 1.281807)], crs=4326
        )
        .to_crs(b_gdf.crs)
        .iloc[0],
        "marina_bay_mrt": gpd.GeoSeries(
            [box(103.853470, 1.270952, 103.860240, 1.276465)], crs=4326
        )
        .to_crs(b_gdf.crs)
        .iloc[0],
    }

    ###################################

    REMOVE_REASONS_DICT = {
        "Landuse-OSM": (buildings_osm_merge["landuse"] == "construction")
        | (buildings_osm_merge["landuse"] == "industrial_port"),
        "Buildingtype-Overture": (b_gdf["subtype"].isin(OVERTURE_REMOVAL_SUBTYPES))
        | (b_gdf["class"].isin(OVERTURE_REMOVAL_CLASSES)),
        "Size-OSM": (b_gdf["src_dataset_label"] == "OpenStreetMap")
        & (b_gdf["area"] < building_size_thresholds["OSM"]),
        "Size-Google": (b_gdf["src_dataset_label"] == "Google Open Buildings")
        & (b_gdf["area"] < building_size_thresholds["Google"]),
        "Size-Microsoft": (b_gdf["src_dataset_label"] == "Microsoft ML Buildings")
        & (b_gdf["area"] < building_size_thresholds["Microsoft"]),
        "Landuse-MP": buildings_mp_merge[
            "landuse"
        ].isna(),  # remove NaNs here, about 130 of them
        "Landuse-MP_coverage": (b_mp_coverage_prop < mp_zone_coverage_threshold)
        & (
            buildings_osm_merge["landuse"] != "residential"
        ),  # prevent removal of very low-density bungalows (using Overture buidling tagging)
        "Bbox-removed-areas": b_gdf.geometry.within(REMOVAL_BBOXES["southern_islands"])
        | b_gdf.geometry.within(REMOVAL_BBOXES["turf_city"])
        | b_gdf.geometry.within(REMOVAL_BBOXES["marina_south"])
        | b_gdf.geometry.within(REMOVAL_BBOXES["marina_bay_mrt"]),
        "Non-OSM_MP-res": (buildings_mp_merge["landuse"] == "RESIDENTIAL")
        & (b_gdf["src_dataset_label"] != "OpenStreetMap"),
        "Building_ID": buildings_mp_merge.index.isin(BUILDING_ID_TO_REMOVE),
    }

    # now asasign all reasons at once
    b_gdf["removal_reason"] = np.select(
        list(REMOVE_REASONS_DICT.values()),
        list(REMOVE_REASONS_DICT.keys()),
        default=None,
    )

    # ALSO Remove LANDUSE-MP ones
    # apply without overwriting non-matches
    b_gdf["removal_reason"] = (
        buildings_mp_merge["landuse"]
        .map(MP_LANDUSE_REMOVAL_DICT)
        .fillna(b_gdf["removal_reason"])
    )

    map_values_b_to_bp(b_gdf, bp_gdf, "removal_reason", "removal_reason")

    print(b_gdf["removal_reason"].value_counts(dropna=False))

    # create boolean removed columns
    b_gdf["is_removed"] = ~b_gdf["removal_reason"].isna()
    bp_gdf["is_removed"] = ~bp_gdf["removal_reason"].isna()

    return b_gdf, bp_gdf

In [25]:
buildings_osm_merge, buildings_mp_merge = merge_with_landuse(b_gdf)

b_gdf1, bp_gdf1 = preprocess_remove_buildings(
    b_gdf,
    bp_gdf,
    buildings_osm_merge,
    buildings_mp_merge,
    mp_landuse_gdf,
)

removal_reason
NaN                      106849
Size-Google               56232
Landuse-MP                40524
Size-Microsoft            18925
Landuse-OSM                4461
Size-OSM                   3593
Buildingtype-Overture      2219
Landuse-MP_coverage        1878
Bbox-removed-areas          623
Non-OSM_MP-res              419
Building_ID                   6
Name: count, dtype: int64


In [26]:
b_gdf1.head(1)

,names,sources,height,min_height,is_underground,num_floors,num_floors_underground,min_floor,geometry,subtype,class,has_parts,src_dataset_label,centroid,area,mp_landuse,mp_gpr,mp_OBJECTID,removal_reason,is_removed
id,,,,,,,,,,,,,,,,,,,,
396eb35f-2fd0-4a23-bd73-c721c53ca662,None,"[{'property': '', 'dataset': 'OpenStreetMap', ...",NaN,NaN,False,NaN,NaN,NaN,"POLYGON ((2736.591 22657.445, 2739.729 22652.4...",NaN,NaN,False,OpenStreetMap,POINT (2739.44 22655.709),17.835138,RESERVE SITE,0.0,619284.0,Size-OSM,True


## Re-add Buildings

see removal visualization from [1_buildings_explore.ipynb](../exploration/1_buildings_explore.ipynb)

In [27]:
def re_add_specified_buildings(
    b_gdf: gpd.GeoDataFrame,
    bp_gdf: gpd.GeoDataFrame,
    buildings_mp_merge,
    building_pts_to_readd: list[Point],
):
    """
    Assigns some specified buildings to be readded.
    `removal_reason` is reset to `NaN`, and these buildings are given a new
    MP plot zoning binding in `buildings_mp_merge` with `landuse=READDED`.
    This is so that they can be included with the processing for buildings with `landuse`
    `OPEN SPACE`, `RESERVE SITE`, etc.

    Returns modified `b_gdf`, `bp_gdf` (with removal reason mapped) and `buildings_mp_merge`.
    """
    b_gdf = b_gdf.copy()
    bp_gdf = bp_gdf.copy()
    buildings_mp_merge = buildings_mp_merge.copy()

    ref_pts = gpd.GeoDataFrame(
        geometry=building_pts_to_readd,
        crs="EPSG:4326",  # longitude/latitude
    ).to_crs(
        "EPSG:3414"
    )  # singapore CRS

    matches = gpd.sjoin(
        b_gdf,
        ref_pts,
        how="inner",
        predicate="contains",
    )

    print(f"{len(matches)} new buildings added.")

    b_gdf.loc[matches.index, "is_removed"] = False
    b_gdf.loc[matches.index, "mp_landuse"] = "READDED"
    buildings_mp_merge.loc[matches.index, "landuse"] = "READDED"

    # print(buildings_mp_merge["landuse"].value_counts())

    map_values_b_to_bp(b_gdf, bp_gdf, "is_removed", "is_removed")

    return b_gdf, bp_gdf, buildings_mp_merge


def re_add_class_buildings(
    b_gdf: gpd.GeoDataFrame,
    bp_gdf: gpd.GeoDataFrame,
    buildings_mp_merge,
    buildings_osm_merge,
    classes_to_readd: list[str],
):
    """
    Assigns buildings with specific `subtype` to be readded.

    Still excludes buildings in construction zones

    And only is available for buildings in certain MP zonings

    Returns modified `b_gdf`, `bp_gdf` (with removal reason mapped) and `buildings_mp_merge`.
    """

    b_gdf = b_gdf.copy()
    bp_gdf = bp_gdf.copy()
    buildings_mp_merge = buildings_mp_merge.copy()

    readd_mask = (
        b_gdf["is_removed"]
        & b_gdf["class"].isin(classes_to_readd)
        & ~buildings_osm_merge["landuse"].isin(["construction"])
        & buildings_mp_merge["landuse"].isin(
            [
                "EDUCATIONAL INSTITUTION",
                "PLACE OF WORSHIP",
                "CIVIC & COMMUNITY INSTITUTION",
                "PARK",
                "BEACH AREA",
                "SPORTS & RECREATION",
            ]
        )
    )
    matches = b_gdf[readd_mask]

    print(f"{len(matches)} new buildings added.")

    b_gdf.loc[matches.index, "is_removed"] = False
    b_gdf.loc[matches.index, "mp_landuse"] = "READDED"
    buildings_mp_merge.loc[matches.index, "landuse"] = "READDED"

    # print(buildings_mp_merge["landuse"].value_counts())

    map_values_b_to_bp(b_gdf, bp_gdf, "is_removed", "is_removed")

    return b_gdf, bp_gdf, buildings_mp_merge

In [28]:
building_points_to_readd = [
    Point(103.989745, 1.360133),  # Changi Jewel
    Point(103.790116, 1.275532),  # PSA Horizons
    Point(103.816265, 1.279862),  # DSTA Tower A
    Point(103.814079, 1.280259),  # DSTA Tower B
    Point(103.764051, 1.366826),  # CMPB
    Point(103.848157, 1.285193),  # Pickering Operation Complex
    Point(103.843409, 1.286274),  # State Courts Towers
    # Point(103.839496, 1.333710),  # MSF Bullding
    Point(103.838981, 1.333015),  # SLF Building
    # Point(103.845037, 1.324116),  # MHA HQ
    Point(103.862737, 1.306581),  # ICA bldg
    Point(103.863469, 1.307150),  # ICA Services ctr
    # SIA Airline House; currently not office but will be in next update
    Point(103.996845, 1.377423),
    # SIA Training Centre; currently not office but will be in next update
    Point(103.965935, 1.345763),
    # changi airport cargo areas, most currently not correctly tagged, but will be in future
    Point(103.993964, 1.374522),  # 1-4
    Point(103.995509, 1.378244),  # 5
    Point(103.996625, 1.380893),  # 6
    Point(103.996260, 1.372398),  # dnata
]

classes_to_readd = ["retail", "supermarket"]

In [29]:
# make sure to verify
b_gdf1, bp_gdf1, buildings_mp_merge = re_add_specified_buildings(
    b_gdf1, bp_gdf1, buildings_mp_merge, building_pts_to_readd=building_points_to_readd
)
b_gdf1, bp_gdf1, buildings_mp_merge = re_add_class_buildings(
    b_gdf1, bp_gdf1, buildings_mp_merge, buildings_osm_merge, classes_to_readd=classes_to_readd
)

18 new buildings added.
32 new buildings added.


## Extra Removals

This is now removal for buildings that I have exact employment data for

In [30]:
def remove_buildings_set_of_points(
    b_gdf: gpd.GeoDataFrame,
    bp_gdf: gpd.GeoDataFrame,
    buildings_mp_merge,
    building_pts_to_remove: list[Point],
    point_crs = "EPSG:4326",
):
    """ """
    b_gdf = b_gdf.copy()
    bp_gdf = bp_gdf.copy()
    buildings_mp_merge = buildings_mp_merge.copy()

    ref_pts = gpd.GeoDataFrame(
        geometry=building_pts_to_remove,
        crs=point_crs,  # longitude/latitude
    ).to_crs(
        "EPSG:3414"
    )  # singapore CRS

    matches = gpd.sjoin(b_gdf, ref_pts, how="inner", predicate="contains")

    print(f"{len(matches)} buildings removed by specific point.")

    b_gdf.loc[matches.index, "is_removed"] = True
    b_gdf.loc[matches.index, "removal_reason"] = "Point_Removal"

    # print(buildings_mp_merge["landuse"].value_counts())

    map_values_b_to_bp(b_gdf, bp_gdf, "is_removed", "is_removed")
    map_values_b_to_bp(b_gdf, bp_gdf, "removal_reason", "removal_reason")

    return b_gdf, bp_gdf, buildings_mp_merge

In [31]:
# Remove GOVERNMENT buildings
gov_gdf = gpd.read_parquet(os.path.join(data_filepaths.path_data_intermediate, "gov_workforce_v1.parquet"))

b_gdf1, bp_gdf1, buildings_mp_merge = remove_buildings_set_of_points(
    b_gdf1,
    bp_gdf1,
    buildings_mp_merge,
    building_pts_to_remove=gov_gdf["geometry"].to_list(),
    point_crs=gov_gdf.crs,
)

19 buildings removed by specific point.


## Formally remove

Retain those with `is_removed == False`

In [32]:
b_gdf_rmv = b_gdf1[~b_gdf1["is_removed"]].drop(columns=["is_removed", "removal_reason"])
bp_gdf_rmv = bp_gdf1[~bp_gdf1["is_removed"]].drop(
    columns=["is_removed", "removal_reason"]
)

print(
    f"{len(b_gdf1) - len(b_gdf_rmv)}/{len(b_gdf1)} buildings removed: "
    + f"{((len(b_gdf1) - len(b_gdf_rmv)) / len(b_gdf1) * 100):.3f}%"
)
print(
    f"{len(bp_gdf1) - len(bp_gdf_rmv)}/{len(bp_gdf1)} building parts removed: "
    + f"{((len(bp_gdf1) - len(bp_gdf_rmv)) / len(bp_gdf1) * 100):.3f}%"
)

128837/235729 buildings removed: 54.655%
1920/8230 building parts removed: 23.329%


# Building Demand Type Classification

## Constants

In [33]:
from src.utils.vis.map_special import build_map_building_demand_type
from enum import StrEnum


class Zoning(StrEnum):
    RES = "RESIDENTIAL"
    RES1C = "RESIDENTIAL WITH COMMERCIAL AT 1ST STOREY"
    RESCOM = "COMMERCIAL & RESIDENTIAL"
    COMRES = "COMMERCIAL & RESIDENTIAL"  # alternative
    COM = "COMMERCIAL"
    HOTEL = "HOTEL"
    WHITE = "WHITE"
    BP = "BUSINESS PARK"
    BPW = "BUSINESS PARK - WHITE"
    B1 = "BUSINESS 1"
    B2 = "BUSINESS 2"
    B1W = "BUSINESS 1 - WHITE"
    B2W = "BUSINESS 2 - WHITE"
    RESINST = "RESIDENTIAL / INSTITUTION"
    COMINST = "COMMERCIAL / INSTITUTION"
    UTILITY = "UTILITY"
    AGRI = "AGRICULTURE"


# include colormaps for demand types
DEMAND_TYPE_COLORMAP = {
    "res_low": "#82c901",
    "res_med": "#019529",
    "res_high": "#05696b",
    "office": "#ed0dd9",
    "commercial": "#5539cc",
    "ind_low": "#fffe7a",
    "ind_med": "#feb308",
    "ind_high": "#ff7855",
    "hotel": "#069af3",
}
DEMAND_TYPE_CMAP_OBJ = CategoricalColorMap(
    cat_colors=list(DEMAND_TYPE_COLORMAP.values()),
    cat_labels=list(DEMAND_TYPE_COLORMAP.keys()),
    nan_color=np.array([0.8, 0.8, 0.8, 0.4]),
)

FLOOR_HEIGHTS = {
    "landed": 3.2,
    "house": 3.2,
    "bungalow": 3.5,
    "detached": 3.5,
    "terrace": 3.2,
    "semidetached_house": 3.2,
    "condo": 3.4,
    "residential": 3,
    "apartments": 3,
    "dormitory": 3,
    "hotel": 3.1,
    "office": 3.9,
    "government": 3.9,
    "commercial": 3.8,
    "retail": 3.8,
    "public": 3.8,
    "supermarket": 4.5,
    "school": 3.4,
    "university": 3.5,
    "hospital": 3.6,
    "industrial": 5,
    "manufacture": 5.2,
    "warehouse": 6,
    # not really gonna mean anything, idt any of them have building:levels anyway:
    "hangar": 6,
    "garage": 2.8,
    "parking": 2.8,
}

RES_GPR_FLOORCOUNT_THRESHOLDS = {1.4: 5, 1.6: 12, 2.1: 24, 2.8: 36}


def clip_res_allowed_floors_gpr(floors, gpr):
    # https://www.ura.gov.sg/guidelines/development-control/development-control-handbooks/residential/flats-condominiums/building-height/
    mapping = RES_GPR_FLOORCOUNT_THRESHOLDS

    for k, v in mapping.items():
        if gpr <= k:
            return min(v, floors)

    return floors

## A - Merge Building Parts

- A: Merge building parts into buildings
    - for every building with parts, IF the building doesn’t have `num_floors` already:
        - note: bparts don’t have `class` and `subtype`
        - account for `min_floor`
        - merge into building based on building’s footprint, assign `num_floors`

In [34]:
def temp_vis_A():
    """
    Visualizes buildings that have parts and num_floors
    """
    temp_b_gdf = b_gdf_rmv.copy()
    temp_bp_gdf = bp_gdf_rmv.copy()

    temp_b_gdf["has_parts_and_num_floors"] = (
        temp_b_gdf["has_parts"] & ~temp_b_gdf["num_floors"].isna()
    )

    print(temp_b_gdf["has_parts_and_num_floors"].sum()) # 323

    map_values_b_to_bp(
        temp_b_gdf, temp_bp_gdf, "has_parts_and_num_floors", "has_parts_and_num_floors"
    )
    map_values_b_to_bp(temp_b_gdf, temp_bp_gdf, "has_parts", "has_parts")

    # cmap = CategoricalColorMap(
    #     cat_colors=list(mp_lu_color_mapping.values()),
    #     cat_labels=list(mp_lu_color_mapping.keys()),
    #     nan_color=np.array([0.2, 0.2, 0.2, 0.5]),
    # )

    mapper = MapExplorer(
        temp_b_gdf,
        temp_bp_gdf,
        # enable_debug_mode=True,
    )
    return mapper.build_singleprop(
        "has_parts_and_num_floors",
        # cmap,
        title="Buildings that have parts and num_floors",
    )

# temp_vis_A()

THERE ARE STILL EXISTING PROBLEMS
* Expo (num_floors = 0) - probably not a big deal?
* Yishun Northpoint (a big deal)

In [35]:
from src.utils.building_tools import merge_building_parts_into_buildings

# print(bp_gdf_rmv["min_height"])

b_gdf2 = merge_building_parts_into_buildings(b_gdf_rmv, bp_gdf_rmv)
print("\n### TOP 5 FLOORCOUNT BUILDINGS:")
print(
    b_gdf2["num_floors"].sort_values(ascending=False).head(5)
)  # should have decimal places
# res[res["extra_floors_times_area"] > 0]


### TOP 5 FLOORCOUNT BUILDINGS:
id
e8090893-44da-4ccd-b08f-625b7e2e6051    107.722730
19734297-5359-4084-9ee4-d60ceef87e2d     92.424063
feca34e6-af9f-4003-a129-8b21339acc66     73.053686
05f6582c-2b94-4c37-a5ee-d9264dbb808b     72.328595
43d2ac74-4c37-48af-bba3-89eb0e561786     71.495930
Name: num_floors, dtype: float64


In [43]:
# MapExplorer(b_gdf2).build_singleprop(
#     "height_calc",
#     title="Buildings with new, combined heights",
#     continuous_cmap_name="YlGnBu",
# )

id
948d28ca-e26d-4536-b48e-dc966c73a7c2    6.0
8f5c1824-e519-4c5b-9d2d-f5bc06777ab2    6.0
67b57e9e-00de-4c08-83c6-bf86bf4b16fd    6.0
c48cd428-246f-4ae5-b406-5a5679e561fb    6.0
5fe7c7ef-e2c3-4648-837f-0367f7a953ce    6.0
Name: height_calc, dtype: float64


## Intermediate: Create new demand-type-building DF | And other setup

And generally cleaning up the dataframe for demand type classification & assignment

In [37]:
# add all MP zonal information to b_gdf
b_gdf_withmp = b_gdf2
b_gdf_withmp[
    ["mp_white_q_max", "mp_gpr_b_min", "mp_is_landed", "mp_is_sdp"]
] = buildings_mp_merge[["white_q_max", "gpr_b_min", "is_landed", "is_sdp"]]

# further streamlining of dataset
b_gdf_withmp = b_gdf_withmp.drop(
    columns=[
        "sources",
        "src_dataset_label",
        "is_underground",
        "min_height",
        "has_parts",
        "names",
    ]
)
b_gdf_withmp["net_floors"] = (
    b_gdf_withmp["num_floors_underground"].fillna(0)
    + b_gdf_withmp["num_floors"]  # this one determines whether it's NaN or not
    - b_gdf_withmp["min_floor"].fillna(0)
)
b_gdf_withmp = b_gdf_withmp.drop(
    columns=["num_floors", "num_floors_underground", "min_floor"]
)

In [38]:
# demandtype buildings gdf
b_dmdtype_gdf = gpd.GeoDataFrame(
    columns=[
        "mp_OBJECTID",
        "mp_landuse",
        "demand_type",
        "geometry",
        "centroid",
        # existing values from building
        "b_id",
        "area",
        "net_floors",
        "height", # to reverse estimate floorcount
        # new adjustment parameters when splitting buildings
        "floorcount_adj",
        "floorspace_coeff",
    ]
)
# new index, not using any existing index

MP columns:

* `mp_OBJECTID`
* `mp_landuse`, `mp_gpr`
* `mp_white_q_max`, `mp_gpr_b_min`, `mp_is_landed`, `mp_is_sdp`

## B(1) - "Future Residential Zone" Verificiation

> This section (B) is technically allocated for all buildings that don't seem to be in the correct MP zonings
> But currently this is only limited to residential zones, that seems the simplest (and **most pressing** to fix)

There are many areas zoned for future residential (`RES`, `RES1C`, `COMRES`) (with GPR >= 2.0),
but currently have much lower-intensity use (anything from sparse industrial land to existing sports 
centres slated for redevelopment).

Note that `RESINST` is omitted. They only occupy some heritage building areas, and won't significant any
new developments.

Couple possible rules. Out of 59797 plots (currently):

* first, no residential in plot
  * 33485 plots
* Additional condition: max floorheight
  * if it's at least 5, that building should be an untagged apartment block
  * At the moment it is *highly unlikely* that some large warehouse with >= 5 floors was built on land designated
    for residential redevelopment.
* if more than *some threshold* covered by building with `industrial`, `warehouse`, `manufacture`
  * = 1.0: 24 plots
  * 0.5 <= x < 1.0: 7 plots
  * pretty easy to decide
* if zone coverage < *some threshold*
  * This likely will be more strict as compared to the initial coverage culling
  * Even when restricted to plots with no residential-tagged buildings, some very sparse landed areas show
    up as false positives, going as low as 7%
  * So further sort based on GPR
    * 1.0 < GPR <= 2.4: use 10%
    * 2.4 < GPR <= 3.0: use 15%
    * 3.0 < GPR: use 20%

Areas where this problem can be found:

* MP Plot zoning already *manually changed*: Admiralty, Jurong (West of Pandan Gardens), Punggol East (693967)
* Mount Pleasant: 589461, 594260, 640035
* Woodlands Av x Admiralty Lane: 671511
* Across the AYE from NUH: 632181 (ex SIT), 645787

What happens afterwards:

* Buildings will be set to `READDED`, cuz zoning & GPR will be unknown
  * (a lot of the time they are construction buildings anyway, so not to be considered)
  * They can then be suitably handled in pt C
  * Yes I am aware that `READDED` is not the best label for this description, but wtv it works

```plaintext
Initial: 84634 buildings, 59797 plots
plots removed:
zonecov_gpr3.0+        118
zonecov_gpr2.4-3.0      73
unwanted_proportion     46
zonecov_gpr1.0-2.4      27
```

A future possible refinement woudl be to check against OSM `landuse=residential`.

In [39]:
def fix_future_residential_zone_buildings(
    b_gdf_withmp: gpd.GeoDataFrame,
    mp_landuse_gdf: gpd.GeoDataFrame,
    debug: bool = False,
):
    b_gdf_withmp = b_gdf_withmp.copy()
    mp_landuse_gdf = mp_landuse_gdf.copy()
    mp_landuse_gdf["area"] = mp_landuse_gdf.area

    b_gdf_res = b_gdf_withmp[
        b_gdf_withmp["mp_landuse"].isin([Zoning.RES, Zoning.RES1C, Zoning.COMRES])
    ]
    num_res_b_initial, num_res_b_plots_initial = len(b_gdf_res), len(
        b_gdf_res["mp_OBJECTID"].unique()
    )

    print(f"Initial: {num_res_b_initial} buildings, {num_res_b_plots_initial} plots")

    # first set up the conditions on each individual building
    b_gdf_res["building_is_res"] = b_gdf_res["subtype"] == "residential"
    b_gdf_res["building_is_unwanted"] = b_gdf_res["subtype"].isin(
        ["industrial", "warehouse", "manufacture"]
    )

    # then groupby and aggregate those
    zones = b_gdf_res.groupby("mp_OBJECTID").agg(
        num_b=("mp_OBJECTID", "size"),
        count_res=("building_is_res", "sum"),
        count_unwanted=("building_is_unwanted", "sum"),
        building_area_sum=("area", "sum"),
        max_floors=("net_floors", "max"),
    )

    zones = zones.merge(
        mp_landuse_gdf[["area", "gpr"]], left_on="mp_OBJECTID", right_index=True
    )

    # calculate desired proportions
    zones["prop_unwanted"] = zones["count_unwanted"] / zones["num_b"]
    zones["prop_area_coverage"] = zones["building_area_sum"] / zones["area"]

    # determine if zones are to be reclassified
    z_gpr = zones["gpr"]
    zones["residential_reclass"] = np.select(
        [
            (zones["count_res"] > 0) | (zones["max_floors"] >= 5),
            zones["prop_unwanted"] >= 0.5,
            (z_gpr > 1) & (z_gpr <= 2.4) & (zones["prop_area_coverage"] < 0.1),
            (z_gpr > 2.4) & (z_gpr <= 3) & (zones["prop_area_coverage"] < 0.15),
            (z_gpr > 3) & (zones["prop_area_coverage"] < 0.2),
        ],
        [
            None,
            "unwanted_proportion",
            "zonecov_gpr1.0-2.4",
            "zonecov_gpr2.4-3.0",
            "zonecov_gpr3.0+",
        ],
        default=None,
    )
    zones["to_reclassify"] = ~zones["residential_reclass"].isna()

    # map back
    reclassify_b_mask = b_gdf_res.merge(
        zones["to_reclassify"], left_on="mp_OBJECTID", right_index=True
    )["to_reclassify"]

    # need to reindex from just residential buildings back to all buildings
    b_gdf_withmp.loc[
        reclassify_b_mask.reindex(b_gdf_withmp.index, fill_value=False), "mp_landuse"
    ] = "READDED"

    if debug:
        # used for debug_analyze_part_c()
        return zones
    else:
        return b_gdf_withmp


def debug_analyze_part_c(show_map: bool):

    out = fix_future_residential_zone_buildings(
        b_gdf_withmp, mp_landuse_gdf, debug=True
    )

    # print((out["prop_unwanted"] > 0.9).sum())
    # print(
    #     out[out["prop_unwanted"] < 0.9]["prop_unwanted"]
    #     .sort_values(ascending=False)
    #     .head(10)
    # )

    # PLOTTING
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 5))

    sns.histplot(
        out, x="prop_unwanted", bins=50, ax=ax1, log_scale=(False, True), fill=False
    )
    sns.histplot(
        out[out["prop_area_coverage"] < 3],
        x="prop_area_coverage",
        stat="proportion",
        ax=ax2,
        cumulative=True,
    )

    b_gdf_withmp_and_conds = b_gdf_withmp.copy()
    b_gdf_withmp_and_conds = b_gdf_withmp_and_conds.merge(
        out, left_on="mp_OBJECTID", right_index=True
    )

    print(b_gdf_withmp_and_conds["residential_reclass"].value_counts())

    def build_map(b_gdf_withmp_and_conds: gpd.GeoDataFrame):
        b_gdf_withmp_and_conds["num_floors"] = b_gdf_withmp_and_conds["net_floors"]
        b_gdf_withmp_and_conds["prop_area_coverage"] = b_gdf_withmp_and_conds[
            "prop_area_coverage"
        ].clip(upper=1)

        mapper = MapExplorer(
            b_gdf_withmp_and_conds[
                (b_gdf_withmp_and_conds["count_res"] == 0)
                & (b_gdf_withmp_and_conds["gpr"] > 1)
            ],
            # temp_bp_gdf,
            # enable_debug_mode=True, # BREAKS WITH DEBUGMODE, IDK WHY
        )

        return mapper.build_configurable(
            ["prop_unwanted", "prop_area_coverage", "residential_reclass"],
            show_histogram=False,
        )

    # print(b_gdf_withmp_and_conds.columns)
    if show_map:
        return build_map(b_gdf_withmp_and_conds)


# debug_analyze_part_c(show_map=True)

b_gdf_withmp = fix_future_residential_zone_buildings(b_gdf_withmp, mp_landuse_gdf)

Initial: 84179 buildings, 59868 plots


## C - Handle Ambiguous MP Plots

> This is where the first demand type allocations are *actually* done.

- `RESERVE SITE`, `OPEN SPACE`, `is_sdp=True`
    - e.g. open_space - a couple of the big buildings at marina south are worth investigating
- Also `READDED`
- select only specific OSM tags, immediately **assign to its demand type**,
ignore others

In [40]:
def handle_ambiguous_mp_plots(
    b_gdf_withmp: gpd.GeoDataFrame, b_dmdtype_gdf: gpd.GeoDataFrame
):
    """
    Returns a `b_dmdtype_gdf` with new appended values

    Only the following Overture building `class` values will be converted to `demand_type`:
    * residential -> res_high
    * apartments -> res_high
    * bungalow -> res_low
    * terrace -> res_low
    * detached -> res_low
    * semidetached_house -> res_low
    * office -> office
    * government -> office
    * public -> commercial
    * commercial -> commercial
    * retail -> commercial
    * supermarket -> commercial
    * industrial -> ind_med
    * manufacture -> ind_med
    * warehouse -> ind_low

    The rest, including NaN (OSM `building=yes`) will be dropped.
    """

    b_selection = b_gdf_withmp[
        b_gdf_withmp["mp_landuse"].isin(["RESERVE SITE", "OPEN SPACE", "READDED"])
        | b_gdf_withmp["mp_is_sdp"]
    ]
    initial_count = len(b_selection)

    b_selection["demand_type"] = b_selection["class"].map(
        {
            "residential": "res_med",
            "apartments": "res_high",
            "bungalow": "res_low",
            "terrace": "res_low",
            "detached": "res_low",
            "house": "res_low",
            "semidetached_house": "res_low",
            "office": "office",
            "government": "office",
            "public": "commercial",
            "commercial": "commercial",
            "retail": "commercial",
            "supermarket": "commercial",
            "industrial": "ind_med",
            "manufacture": "ind_med",
            "warehouse": "ind_low",
            "hangar": "ind_low",
        }
    )
    new_count = b_selection["demand_type"].count()
    # down to 400 selected buildings lmao
    print(
        f"Out of {initial_count} total ambiguous buildings, "
        + f"{new_count} buildings ({(new_count/initial_count*100):.2f}%) successfully tagged with demand_type."
    )

    b_final = b_selection.copy()
    # b_final = b_selection[~b_selection["demand_type"].isna()]

    floor_heights_series = b_final["class"].map(FLOOR_HEIGHTS)

    to_append = gpd.GeoDataFrame(
        {
            "mp_OBJECTID": b_final["mp_OBJECTID"],
            "mp_landuse": b_final["mp_landuse"],
            "demand_type": b_final["demand_type"],
            "geometry": b_final["geometry"],
            "centroid": b_final["centroid"],
            "b_id": b_final.index,
            "area": b_final["area"],
            # TODO might wanna improve, i.e. based on demand type
            # currently takes from height, assumes avr floor height taken from FLOOR_HEIGHTS dict
            # and if nothing is specified, gives it a default of 2 floor
            "net_floors": b_final["net_floors"].fillna(
                (b_final["height"] / floor_heights_series).fillna(2)
            ),
            "height": b_final["height"],
            "floorcount_adj": 0,
            "floorspace_coeff": 1,
        }
    )

    return pd.concat([b_dmdtype_gdf, to_append])


# .head(0) - this assumes empty starting df, so just take the cols
res = handle_ambiguous_mp_plots(b_gdf_withmp, b_dmdtype_gdf.head(0))
b_dmdtype_gdf = res[~res["demand_type"].isna()]

# this does not show buildings that DID NOT get tagged... for some reason
# I literally cannot get res to show as well, some very weird error
# build_map_building_demand_type(b_dmdtype_gdf, DEMAND_TYPE_CMAP_OBJ)

Out of 3667 total ambiguous buildings, 912 buildings (24.87%) successfully tagged with demand_type.


In [41]:
b_gdf_withmp[b_gdf_withmp.index == "19c2b793-97f1-4462-b15a-fb5cbf0122cd"]

,height,geometry,subtype,class,centroid,area,mp_landuse,mp_gpr,mp_OBJECTID,mp_white_q_max,mp_gpr_b_min,mp_is_landed,mp_is_sdp,net_floors
id,,,,,,,,,,,,,,


## D - Handle main Building Classification

Floor heights are filled in for those buildings with known OSM taggings 
(more specific than `demand_type`)

This is a departure for the original pipeline that would have calculated

Buildings with no floors specified here will have it extrapolated from total
plot GFA in the floorspace estimation step.

In [42]:
def handle_main_zoned_buildings(b_gdf_withmp, b_dmdtype_gdf):
    b_gdf_withmp = b_gdf_withmp.copy()
    # specifically exclude those from part C
    b_gdf_withmp = b_gdf_withmp[
        ~b_gdf_withmp["mp_landuse"].isin(["RESERVE SITE", "OPEN SPACE", "READDED"])
        & ~b_gdf_withmp["mp_is_sdp"]
    ]

    b_dmdtype_gdf = b_dmdtype_gdf.copy()  # output to be concated to

    residential_mask = b_gdf_withmp["mp_landuse"] == Zoning.RES
    b_gdf_res = b_gdf_withmp[residential_mask]
    b_gdf_nonres = b_gdf_withmp[~residential_mask]

    new_rows = []  # dict matching output schema

    def create_new_b_dmdtype_row_dict(
        index,
        row,
        demand_type: str,
        building_class: str = None,  # for floor height calculation
        floors: float = None,
        floorcount_adj=0,
        floorspace_coeff=1.0,
        floor_ht_default=3.5,
    ):
        # row: existing b_gdf_withmp row

        # net floor count
        if floors is not None:
            # had overriding floors
            floors = floors
        elif not pd.isna(row["net_floors"]):
            # has floor values
            floors = row["net_floors"]
        elif not pd.isna(row["height"]):
            # doesn't have floor values but has height
            building_class = building_class if building_class else row["class"]
            floors = (
                (row["height"] / FLOOR_HEIGHTS[building_class])
                if (building_class in FLOOR_HEIGHTS)
                else (row["height"] / floor_ht_default)
            )
        else:
            floors = None

        return {
            "mp_OBJECTID": row["mp_OBJECTID"],
            "mp_landuse": row["mp_landuse"],
            "demand_type": demand_type,
            "geometry": row["geometry"],
            "centroid": row["centroid"],
            "b_id": index,
            "area": row["area"],
            "net_floors": floors,
            "height": row["height"],
            "floorcount_adj": floorcount_adj,
            "floorspace_coeff": floorspace_coeff,
        }

    add_row = lambda row, *args, **kwargs: new_rows.append(
        create_new_b_dmdtype_row_dict(row, *args, **kwargs)
    )

    for index, row in b_gdf_res.iterrows():
        add_helper = lambda *args, **kwargs: add_row(index, row, *args, **kwargs)

        # first handle non-residential possibilities
        if row["class"] in ("commercial", "retail", "supermarket"):
            add_helper("commercial", "commercial")
        elif row["class"] == "office":
            add_helper("office", "office")
        elif row["subtype"] == "industrial":
            add_helper("ind_low", floor_ht_default=FLOOR_HEIGHTS["industrial"])
        elif row["mp_is_landed"]:
            add_helper("res_low", "landed")
        elif row["mp_gpr"] <= 2:
            add_helper("res_med", "residential")
        else:
            add_helper("res_high", "residential")

    for index, row in b_gdf_nonres.iterrows():
        add_helper = lambda *args, **kwargs: add_row(index, row, *args, **kwargs)
        landuse = row["mp_landuse"]
        b_class = row["class"]
        b_subtype = row["subtype"]

        # first handle specific building types:
        if b_class == "warehouse":
            # warehouses are ind_low
            add_helper("ind_low", floor_ht_default=FLOOR_HEIGHTS["warehouse"])
            continue
        elif b_class in ("retail", "supermarket"):
            add_helper("commercial", "commercial")
            continue

        if landuse == Zoning.RES1C:
            add_helper("commercial", "commercial", floors=1)
            add_helper("res_high", "residential", floorcount_adj=-1)
        elif landuse == Zoning.COMRES:
            add_helper("commercial", "commercial", floorspace_coeff=0.35)
            add_helper("res_high", "residential", floorspace_coeff=0.65)
        elif landuse == Zoning.COM:
            if b_class == "office":
                add_helper("office", "office")
                continue
            # some residential towers exist on commercial plots
            if b_class in ("residential", "apartments"):
                add_helper("res_high", "residential")

            add_helper("commercial", floor_ht_default=FLOOR_HEIGHTS["commercial"])
        elif landuse == Zoning.HOTEL:
            add_helper("hotel", "hotel", floorspace_coeff=0.7)
            add_helper("commercial", "commercial", floorspace_coeff=0.3)
        elif landuse == Zoning.WHITE:
            # ah, the painfully ambiguous one
            # break it down by building
            if b_class in ("commercial", "office"):
                add_helper(b_class, b_class)
            elif b_class in ("residential", "apartments"):
                add_helper("res_high", "residential")
            elif b_class == "hotel":
                add_helper("hotel", "hotel", floorspace_coeff=0.7)
                add_helper("commercial", "commercial", floorspace_coeff=0.3)
            else:
                # default WHITE version
                add_helper("office", "office", floorspace_coeff=0.6)
                add_helper("commercial", "commercial", floorspace_coeff=0.4)

        elif landuse == Zoning.BP:
            if b_class == "hotel":
                # there are a few cases of these
                add_helper("hotel", "hotel")
                continue

            add_helper("office", "office")
        elif landuse == Zoning.BPW:
            w_pct = (
                row["mp_white_q_max"] / 100
                if not pd.isna(row["mp_white_q_max"])
                else 0.3
            )
            add_helper("office", "office", floorspace_coeff=1 - w_pct)
            add_helper("commercial", "commercial", floorspace_coeff=w_pct)

        elif landuse == Zoning.B1 or landuse == Zoning.B2:
            # if commercial buildings occupy B1/B2 zoned plots
            if b_class in ("commercial", "office"):
                # 0.67 floorspace multiplier for commercial/office buildings in industrial (B1/B2) zones
                add_helper(b_class, b_class, floorspace_coeff=0.67)
                continue

            gpr = row["mp_gpr"]
            if gpr <= 1:
                add_helper("ind_low", floor_ht_default=FLOOR_HEIGHTS["warehouse"])
            elif gpr <= 2:
                add_helper("ind_med", floor_ht_default=FLOOR_HEIGHTS["industrial"])
            else:
                add_helper("ind_high", floor_ht_default=FLOOR_HEIGHTS["industrial"])
        elif landuse == Zoning.B1W or landuse == Zoning.B2W:
            w_pct = (
                row["mp_white_q_max"] / 100
                if not pd.isna(row["mp_white_q_max"])
                else 0.3
            )
            add_helper(
                "ind_med",
                floor_ht_default=FLOOR_HEIGHTS["industrial"],
                floorspace_coeff=1 - w_pct,
            )
            add_helper("commercial", "commercial", floorspace_coeff=w_pct)

        elif landuse == Zoning.RESINST:
            add_helper("res_med", "residential")
        elif landuse == Zoning.COMINST:
            add_helper("commercial", "commercial")
        elif landuse == Zoning.AGRI:
            add_helper("ind_low", "industrial")
        elif landuse == Zoning.UTILITY:
            add_helper("ind_low", "industrial")
        else:
            # There wouldn't be any left here right?
            # unless I introduce a specific "uncategorized" type; then like subtype=entertainment could get it here
            pass

    new_df = gpd.GeoDataFrame(new_rows, columns=b_dmdtype_gdf.columns, crs=3414)
    out = pd.concat([b_dmdtype_gdf, new_df], ignore_index=True)

    return out


b_dmdtype_gdf = handle_main_zoned_buildings(
    b_gdf_withmp,
    b_dmdtype_gdf,
    # b_dmdtype_gdf.head(0),
)
# b_dmdtype_gdf.head(5)

build_map_building_demand_type(b_dmdtype_gdf, DEMAND_TYPE_CMAP_OBJ)

# Building Floorspace Estimation

Output:

* Produces a final `b_pt_gdf` where `geometry` is just inidividual points
* Visualizations now become points with radius rather than building footprints with heights

In [44]:
mp_landuse_gdf.columns

Index(['landuse', 'gpr', 'white_q_max', 'gpr_b_min', 'geometry', 'is_landed',
       'is_sdp', 'color'],
      dtype='str')

In [45]:
mp_landuse_gdf[
    (mp_landuse_gdf["gpr"] == 0.0)
    & (mp_landuse_gdf["landuse"] == "RESIDENTIAL")
    & ~mp_landuse_gdf["is_landed"]
]
# ]["landuse"].value_counts()

,landuse,gpr,white_q_max,gpr_b_min,geometry,is_landed,is_sdp,color
OBJECTID,,,,,,,,
692409,RESIDENTIAL,0.0,NaN,NaN,"POLYGON ((29418.443 26153.263, 29419.22 26162....",False,False,"(0.9568627450980393, 0.7294117647058823, 0.509..."
692410,RESIDENTIAL,0.0,NaN,NaN,"POLYGON ((29302.554 25564.254, 29264.736 25590...",False,False,"(0.9568627450980393, 0.7294117647058823, 0.509..."
692411,RESIDENTIAL,0.0,NaN,NaN,"POLYGON ((29148.194 25894.947, 29146.817 25895...",False,False,"(0.9568627450980393, 0.7294117647058823, 0.509..."
692647,RESIDENTIAL,0.0,NaN,NaN,"POLYGON ((21230.153 46116.995, 21223.102 46157...",False,True,"(0.9568627450980393, 0.7294117647058823, 0.509..."
692486,RESIDENTIAL,0.0,NaN,NaN,"POLYGON ((28643.202 31847.73, 28643.192 31847....",False,False,"(0.9568627450980393, 0.7294117647058823, 0.509..."
...,...,...,...,...,...,...,...,...
689038,RESIDENTIAL,0.0,NaN,NaN,"POLYGON ((29472.868 25941.721, 29473.259 25942...",False,False,"(0.9568627450980393, 0.7294117647058823, 0.509..."
689039,RESIDENTIAL,0.0,NaN,NaN,"POLYGON ((29325.374 26057.27, 29310.775 26084....",False,False,"(0.9568627450980393, 0.7294117647058823, 0.509..."
689040,RESIDENTIAL,0.0,NaN,NaN,"POLYGON ((27964.195 24544.611, 27959.171 24544...",False,False,"(0.9568627450980393, 0.7294117647058823, 0.509..."


Note: this next function, because it's not vectorized at all 
(I'm not smart enough to figure that out), takes ages to run. 
\~3m 30s with `fs_calc` logging, 
way more (\~) with `fs_calc` logging

In [46]:
def calc_floorspace(
    b_dmdtype_gdf: gpd.GeoDataFrame,
    mp_landuse_gdf: gpd.GeoDataFrame,
    default_building_floorcount: float = 1,
    res_gpr_floorcount_thresholds: dict = RES_GPR_FLOORCOUNT_THRESHOLDS,
    max_gpr_ratio_error_correction: float = 3,
    debug_building_ids: list[str] = [],
    debug_log_fs_calc: bool = False,
):
    """
    Returns a new building demandtype gdf with new column `floorspace`.
    And new column `fs_calc` that documents which operation(s) produced the floorspace, if `debug_log_fs_calc` is enabled.

    Needs `mp_landuse_gdf` from a prior step, with correct objectid index,
    and columns `gpr`, `is_landed`, and its polygon `geometry`.

    Plot GFA: gross floor area, `plot area * GPR`

    ---

    TODO description needs to be improved

    There are two types of plots, standard demand plots, and ambiguous plots plots.
    Standard demamd plots are self-explanatory. ambiguous plots are those plots
    that would have originally been removed, because: they were from zonings that I don't want to calculate demand for,
    but that have notable commercial/office buildings that still should be included; or simply ambiguous zones.
    `is_sdp`, `mp_landuse=READDED, RESERVE SITE, OPEN SPACE`.

    Floorspace calculation is only done in two principal ways, with two conditions:
    * Floor count is already known. This is either an actual OSM `building:levels`, or extrapolated in an earlier step.
    Calculate `floorspace` = `net_floors` * `area`
    * Floor count is not known. GFA-remainder-distribution occurs: For all remaining buildings with no known floor
    count, remaining plot GFA after accounting for calculated buildings is distributed.
        * Buildings in `READDED` without known floor counts, with no GPR to serve as an upper bound, are
        set with a default of 1 floor (set in param `default_building_floors`).

    Plot GFA bounding
    * The plot max GFA is used as an upper bound for building floorspaces in a plot. This is to prevent excessive
    overcounting if building footprints overlapped. If the calculated total floorspace exceeds max plot GFA, it
    will be linearly scaled down based on building footprint areas.
        * This can only happen if **all** buildings in a plot *have* existing floorcounts, as
        otherwise, the prior GFA-remainder-distribution step will have handled them already.

    ---
    `fs_calc` values:
    *
    """

    b_dmdtype_gdf = b_dmdtype_gdf.copy()
    # b_dmdtype_gdf["net_floors"] = b_dmdtype_gdf["net_floors"].fillna(default_building_floors)
    mp_landuse_gdf = mp_landuse_gdf[["gpr", "geometry", "is_landed", "is_sdp"]].copy()
    mp_landuse_gdf["mp_area"] = mp_landuse_gdf.area

    buildings_with_mp = b_dmdtype_gdf.merge(
        mp_landuse_gdf, left_on="mp_OBJECTID", right_index=True
    )

    ## vectorized ops to process on entire buildings GDF first
    # setup empty fs_calc col
    buildings_with_mp["fs_calc"] = ""

    # boolean masks
    buildings_with_mp["_has_fc"] = ~buildings_with_mp["net_floors"].isna()
    buildings_with_mp["_is_res"] = buildings_with_mp["demand_type"].isin(
        ["res_low", "res_med", "res_high"]
    )

    # residential floorcount based on GPR thresholds
    # but I apply to the whole GDF anyway, conditioning on res is only checked later
    fc_thres_keys = np.array(sorted(res_gpr_floorcount_thresholds))
    fc_thres_values = np.array(
        [res_gpr_floorcount_thresholds[k] for k in fc_thres_keys]
    )
    idx = np.searchsorted(fc_thres_keys, buildings_with_mp["gpr"], side="left")
    buildings_with_mp["_max_res_fc"] = np.where(
        idx < len(fc_thres_keys),
        fc_thres_values[np.minimum(idx, len(fc_thres_keys) - 1)],
        buildings_with_mp[
            "net_floors"
        ],  # keep default net_floors when bigger than largest GPR
    )
    # not ideal, but I have to fill unavailable values with something, might as well use the default fc here
    buildings_with_mp["_max_res_fc"] = buildings_with_mp["_max_res_fc"].fillna(
        default_building_floorcount
    )

    def process_groups(group):
        """
        Processes a group of buildings associated with each plot.

        Returns a dataframe with row-wise `floorspace`, and `fs_calc` (which calculation took place)
        """

        def _append_fs_calc(string, mask=None):
            if not debug_log_fs_calc:
                return

            if not string.startswith(" "):
                string = " " + string

            if mask is None:
                group["fs_calc"] = group["fs_calc"] + string
                return
            group.loc[mask, "fs_calc"] = group[mask]["fs_calc"] + string

        if (
            group["mp_landuse"].iloc[0] == "READDED"
            or group["mp_landuse"].iloc[0] == "RESERVE SITE"
            or group["mp_landuse"].iloc[0] == "OPEN SPACE"
            or group["is_sdp"].iloc[0]
        ):
            # easiest, this can only happen if buildings have existing floor counts
            # otherwise, there is no plot GPR to calculate from.
            # those buildings are simple not processed :)

            group["floorspace"] = (
                group["net_floors"].fillna(default_building_floorcount) * group["area"]
            )
            _append_fs_calc("ambig_bldg_w_fc")

            # debug thingy
            if (group["b_id"].isin(debug_building_ids)).any():
                print(f"\nMP Area: {group["mp_area"].iloc[0]}")
                print(f"MP GPR: {group["gpr"].iloc[0]}")
                print(group[["demand_type", "area", "net_floors", "floorspace"]].head())
            return group[["floorspace", "fs_calc"]]

        # non READDED
        fc_mask = group["_has_fc"]
        grp_with_fc = group[fc_mask]
        grp_without_fc = group[~fc_mask]

        res_mask = group["_is_res"]
        res_without_fc_mask = res_mask & ~fc_mask

        # GFAs that are zero - replace with 1
        max_gfa = (
            group["mp_area"].iloc[0]
            * (1 if group["gpr"].iloc[0] == 0 else group["gpr"].iloc[0])
            if not group["is_landed"].iloc[0]
            else group["mp_area"].iloc[0]
        )

        # first calculate those with floorcounts
        group.loc[fc_mask, "floorspace"] = (
            grp_with_fc["net_floors"] * grp_with_fc["area"]
        ) * grp_with_fc["floorspace_coeff"] + grp_with_fc["area"] * grp_with_fc[
            "floorcount_adj"
        ]
        _append_fs_calc("w_fc", mask=fc_mask)

        # now handle cases where buildings without floorcount are found
        if len(grp_without_fc) > 0:
            # clip by 0 -> there can exist situations where existing buildings exceed max_gfa anyway,
            # if that happens, no more floorspace is distributed
            remaining_gfa = max(max_gfa - group["floorspace"].sum(), 0)

            # special case: there aren't any buildings *with* floors in the first place
            # I decide to institute a 0.9 coefficient on available floorspace
            # It's likely to be poorly-documented land, so this is some sort of a safety checker
            if len(grp_with_fc) == 0:
                remaining_gfa = 0.9 * max_gfa
                _append_fs_calc("all_no_fc_gpr_limited", mask=~fc_mask)

            group.loc[~fc_mask, "floorspace"] = (
                remaining_gfa
                * (grp_without_fc["area"] / grp_without_fc["area"].sum())
                * grp_without_fc["floorspace_coeff"]
                + grp_without_fc["area"] * grp_without_fc["floorcount_adj"]
            )
            _append_fs_calc("gpr_extrapolate", mask=~fc_mask)

            # residential-only floorcount clipping, based on the residential max floorcounts calculated earlier
            b_res = group[res_without_fc_mask]
            b_res_calculated_fc = b_res["floorspace"] / b_res["area"]

            # log first before floorspace gets updated
            _append_fs_calc(
                "res_fc_gpr_capped",
                mask=res_without_fc_mask
                & ((group["floorspace"] / group["area"]) > group["_max_res_fc"]),
            )

            # apply back to group selection
            group.loc[res_without_fc_mask, "floorspace"] = (
                np.minimum(b_res_calculated_fc, b_res["_max_res_fc"]) * b_res["area"]
            )

        # debug checker
        if (group["b_id"].isin(debug_building_ids)).any():
            print(f"\nMP Area: {group["mp_area"].iloc[0]}")
            print(f"MP GPR: {group["gpr"].iloc[0]}")
            print(f"Max GFA: {max_gfa}")
            print(group[["demand_type", "area", "net_floors", "floorspace"]].head())

        calculated_area_sum = group["floorspace"].sum()
        if calculated_area_sum > max_gfa:
            # if adj_coeff is above a value, assume that it's an error (wrong plot assigned), then keep original value
            adj_coeff = calculated_area_sum / max_gfa

            if adj_coeff <= max_gpr_ratio_error_correction:
                group["floorspace"] = group["floorspace"] / (adj_coeff)
        return group[["floorspace", "fs_calc"]]

    ## GROUP and APPLY
    buildings_with_mp_grouped = buildings_with_mp.groupby(
        "mp_OBJECTID",
        group_keys=False,
    )

    b_dmdtype_gdf[["floorspace", "fs_calc"]] = buildings_with_mp_grouped.apply(
        process_groups
    )
    # b_dmdtype_gdf["floorspace"] = floorspaces.to_numpy().astype(np.float32)
    b_dmdtype_gdf["floorspace"] = b_dmdtype_gdf["floorspace"].astype(np.float32)
    return b_dmdtype_gdf


b_dmdtype_gdf_w_floorspace = calc_floorspace(
    # b_dmdtype_gdf.iloc[:2000],
    # b_dmdtype_gdf[b_dmdtype_gdf["mp_OBJECTID"] == 645903],
    b_dmdtype_gdf,
    mp_landuse_gdf,
    debug_building_ids=["a27f2910-e56e-49cf-92ef-c7c1d8cc1de0"],
    # debug_log_fs_calc=True,
    debug_log_fs_calc=False,
)


MP Area: 483173.0432013594
MP GPR: 0.05
Max GFA: 24158.65216006797
       demand_type        area net_floors  floorspace
107575     ind_low  153.213293        NaN  227.911829
107576     ind_low    391.8471        NaN  582.890606
107577     ind_low  413.600004        NaN  615.249052
107578     ind_low  146.573861        NaN  218.035368
107579     ind_low  167.805173        NaN  249.617922


In [47]:
# finally remove geometry and replace with centroid
def change_geometry_to_centroid(gdf):
    """
    Assumes that both `geometry` and `centroid` columns are found.
    """
    gdf = gdf.copy()
    crs = gdf.crs
    
    gdf = gdf.drop(columns=["geometry"])
    gdf = gdf.rename(columns={"centroid": "geometry"})
    gdf = gdf.set_geometry("geometry").set_crs(crs)
    return gdf

b_pt_gdf = change_geometry_to_centroid(b_dmdtype_gdf_w_floorspace)

#### Finally, Visualize

In [48]:
b_pt_gdf.sort_values("floorspace", ascending=False).head(5)

,mp_OBJECTID,mp_landuse,demand_type,geometry,b_id,area,net_floors,height,floorcount_adj,floorspace_coeff,floorspace,fs_calc
82601,606343.0,BUSINESS 2,ind_high,POINT (8442.744 32973.299),ca8c3bbb-f3b0-4450-84ac-58e430b83518,29503.268258,20.0,NaN,0,1.0,590065.37500,
104689,602077.0,BUSINESS 2,ind_high,POINT (38527.604 38082.037),a405b078-a761-4d19-b6f3-e4d3f374b8d9,27710.949778,NaN,NaN,0,1.0,537468.93750,
92506,600806.0,COMMERCIAL,commercial,POINT (29593.506 28900.521),e8090893-44da-4ccd-b08f-625b7e2e6051,4201.346148,107.72273,NaN,0,1.0,452580.46875,
83817,607254.0,BUSINESS 2,ind_high,POINT (10686.257 32910.121),360c67a7-9b59-43d6-89b3-c7f98a861a09,44642.87001,10.0,NaN,0,1.0,446428.68750,
87108,600412.0,BUSINESS 2,ind_low,POINT (15650.857 27683.992),2fd93de8-9c67-4346-b9b2-ce40d609b914,36755.124215,NaN,NaN,0,1.0,348973.21875,


In [49]:
from src.utils.vis.map_special import build_map_point_floorspace_demand_type

build_map_point_floorspace_demand_type(
    b_pt_gdf[b_pt_gdf["floorspace"] > 0], cmap=DEMAND_TYPE_CMAP_OBJ
)

# Export

In [ ]:
# assert False, "EXPORT PREVENTED BECAUSE DEBUG IN PROCESS"

def export_b_pt_gdf(b_pt_gdf: gpd.GeoDataFrame, fname: str = "buildings.parquet"):
    b_pt_gdf = b_pt_gdf[b_pt_gdf["floorspace"] > 0] # why is this required, I don't remember
    b_pt_gdf = b_pt_gdf[["demand_type", "geometry", "area", "floorspace", "mp_OBJECTID", "mp_landuse"]]

    b_pt_gdf.to_parquet(os.path.join(data_filepaths.path_data_intermediate, fname))

    return b_pt_gdf

if ALLOW_EXPORT:
    export_b_pt_gdf(b_pt_gdf, "buildings_v2.parquet")

,demand_type,geometry,area,floorspace,mp_OBJECTID,mp_landuse
0,commercial,POINT (5984.509 36097.087),3813.386378,7626.772949,610952.0,READDED
1,ind_med,POINT (12436.464 37477.536),4524.859129,27149.154297,607491.0,BUSINESS PARK
2,ind_med,POINT (12312.568 37379.005),2250.115225,9000.460938,612229.0,BUSINESS PARK
3,office,POINT (12255.523 37217.486),8831.657821,35046.363281,612230.0,BUSINESS PARK
4,office,POINT (12251.137 37326.453),634.307388,1268.614746,612230.0,BUSINESS PARK
...,...,...,...,...,...,...
108642,ind_med,POINT (46828.836 39983.936),8200.082561,22934.537109,590958.0,BUSINESS 2
108643,commercial,POINT (46795.375 40071.212),7109.360988,19053.087891,590958.0,BUSINESS 2
108645,ind_med,POINT (46600.914 40186.962),15449.575522,51638.031250,602373.0,BUSINESS 2
108646,ind_med,POINT (46671.315 40401.46),13841.906194,31054.025391,662856.0,BUSINESS 2
